# Chapter 1 - Lab 3: Non-agentic vs agentic workflow (customer-care edition)

**Goal**: compare three ways of answering *"Where is my order?"*, the most common customer-care question:

1. **Non-agentic**: a plain LLM call, with no access to company systems.
2. **Manual tool calling**: we describe a tool to the model, the model asks for it, *our code* runs it.
3. **Agent (OpenAI Agents SDK)**: the framework runs the *think -> call tool -> read result -> answer* loop for us.

*Finance version*: "What is the latest price of NVIDIA?" with a yfinance tool, plus a news agent.
*Customer-care version*: "Where is my order ORD-78120?" with an order-management tool, plus a
service-alerts agent (courier delays, outages, weather).

**Tech stack**: OpenAI Responses API (function calling), OpenAI Agents SDK. The order system and
alert feed are **mock data** defined in this notebook, so no extra API keys are needed.

In [ ]:
%pip install -q openai openai-agents

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

## Mock company systems

In [ ]:
from datetime import date, timedelta

TODAY = date.today()

# Mock order-management system (OMS). In production this would be an API call to your OMS.
ORDERS = {
    "ORD-78120": {"customer_id": "C1002", "product": "Zenith X1 Smartphone", "amount_inr": 34999,
                  "status": "IN_TRANSIT", "last_scan": "Delhi (Okhla) sorting hub",
                  "promised_date": (TODAY - timedelta(days=6)).isoformat(),
                  "current_eta": (TODAY + timedelta(days=2)).isoformat(),
                  "courier": "SwiftShip", "delay_reason": "Backlog at Delhi hub after festive-season surge"},
    "ORD-77812": {"customer_id": "C1001", "product": "FrostLine Refrigerator 340L", "amount_inr": 62990,
                  "status": "DELIVERED", "last_scan": "Delivered, Pune",
                  "promised_date": (TODAY - timedelta(days=41)).isoformat(),
                  "current_eta": None, "courier": "NovaMart Logistics", "delay_reason": None},
    "ORD-78305": {"customer_id": "C1003", "product": "AquaPure RO Purifier", "amount_inr": 18500,
                  "status": "OUT_FOR_DELIVERY", "last_scan": "Chennai (Guindy) delivery centre",
                  "promised_date": TODAY.isoformat(), "current_eta": TODAY.isoformat(),
                  "courier": "SwiftShip", "delay_reason": None},
}

# Mock service-status feed (courier hubs, outages, weather). Replaces the NewsAPI feed of the finance lab.
SERVICE_ALERTS = [
    {"date": (TODAY - timedelta(days=1)).isoformat(), "city": "Delhi", "severity": "HIGH",
     "title": "SwiftShip Delhi (Okhla) hub running 4-6 days behind",
     "detail": "Festive-season volume is 2.3x normal; NovaMart has added two extra line-haul trucks."},
    {"date": (TODAY - timedelta(days=2)).isoformat(), "city": "Chennai", "severity": "MEDIUM",
     "title": "Heavy rain alert: installations may be rescheduled",
     "detail": "Outdoor AC installations in Chennai are paused for 48 hours for technician safety."},
    {"date": TODAY.isoformat(), "city": "Pune", "severity": "LOW",
     "title": "Pune warehouse back to normal after inventory audit",
     "detail": "Large-appliance dispatch resumed; replacement units ship within 72 hours."},
    {"date": (TODAY - timedelta(days=3)).isoformat(), "city": "All", "severity": "MEDIUM",
     "title": "WhatsApp support response times elevated",
     "detail": "Median first response 3.5 hours vs 1 hour target due to a CRM migration."},
]
print("Mock systems loaded. Today is", TODAY)

# 1. Non-agentic workflow

In [ ]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(model="gpt-4.1", input="Where is my NovaMart order ORD-78120 and when will it arrive?")
print(response.output_text)

### Another try: give it a tracking link

In [ ]:
response2 = client.responses.create(
    model="gpt-4.1",
    input="Where is my order ORD-78120? You can check https://track.novamart.example/ORD-78120",
)
print(response2.output_text)

Same problem: a plain API call cannot browse or query company systems. It either declines or guesses.

# 2. Agentic workflow: tool calling without a framework

Three steps:
1. **Describe** the tool to the model (name, purpose, parameters as a JSON schema).
2. **Run** the tool in *our* code when the model asks for it.
3. **Return** the result to the model so it can write the answer.

The model never touches the database. It only *requests* a call, and our code decides whether to run it.
That separation is the foundation of safe customer-care automation.

In [ ]:
import json

# 1. Describe the tool
tools = [
    {
        "type": "function",
        "name": "get_order_status",
        "description": "Look up the live delivery status of a NovaMart order.",
        "parameters": {
            "type": "object",
            "properties": {"order_id": {"type": "string", "description": "Order ID such as ORD-78120"}},
            "required": ["order_id"],
        },
    },
]

# 2. Implement the tool (a mock OMS lookup)
def get_order_status(order_id: str) -> dict:
    order = ORDERS.get(order_id.strip().upper())
    if order is None:
        return {"error": f"No order found with ID {order_id}"}
    return {"order_id": order_id, **order, "as_of": TODAY.isoformat()}

input_list = [{"role": "user", "content": "Where is my order ORD-78120 and when will it arrive?"}]

response = client.responses.create(model="gpt-4.1-mini", tools=tools, input=input_list)
print(response.output)

The model did **not** answer. It returned a `function_call` item with the tool name and the argument it extracted (`ORD-78120`). Now our code executes it:

In [ ]:
input_list += response.output   # keep the model's tool request in the conversation

for item in response.output:
    if item.type == "function_call" and item.name == "get_order_status":
        args = json.loads(item.arguments)
        result = get_order_status(**args)
        print("Tool result:", result)
        input_list.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": json.dumps(result),
        })

In [ ]:
# 3. Send the tool result back so the model can write the customer-facing answer
response = client.responses.create(
    model="gpt-4.1-mini",
    instructions="You are NovaMart's care assistant. Answer only from the tool result. "
                 "Apologise for delays, give the new ETA and the reason in one short paragraph.",
    tools=tools,
    input=input_list,
)
print(response.output_text)

# 3. OpenAI Agents SDK: the framework runs the loop

An **agent** = instructions + model + tools. `Runner` does the loop we wrote by hand above.

In [ ]:
from agents import Agent, Runner, function_tool
# Jupyter/Colab already run an event loop, so cells use "await Runner.run(...)".
# (In a plain .py script you would call Runner.run_sync(...) instead.)

## Order Status Agent

In [ ]:
@function_tool
def lookup_order(order_id: str) -> dict:
    """Look up the live delivery status of a NovaMart order (order IDs look like ORD-78120)."""
    return get_order_status(order_id)

order_status_agent = Agent(
    name="Order Status Agent",
    instructions=("You are NovaMart's order-tracking assistant. Always use the lookup_order tool; "
                  "never guess a date. Be empathetic if the order is late."),
    model="gpt-4.1",
    tools=[lookup_order],
)

output = await Runner.run(order_status_agent, "What is today's date, and where is my order ORD-78120?")
print(output.final_output)

## Service Alerts Agent (replaces the finance News Agent)

Two tools:
* `search_service_alerts` returns raw alerts for a city from the status feed.
* `summarize_service_alerts` fetches the alerts **and** asks an LLM to summarise them: a tool can itself call an LLM.

In [ ]:
def _search_alerts(city: str, max_alerts: int = 3) -> list[dict]:
    city = city.strip().title()
    hits = [a for a in SERVICE_ALERTS if a["city"] in (city, "All")]
    return sorted(hits, key=lambda a: a["date"], reverse=True)[:max_alerts]

@function_tool
def search_service_alerts(city: str, max_alerts: int = 3) -> str:
    """Get the most recent service alerts (courier delays, outages, weather) for an Indian city."""
    alerts = _search_alerts(city, max_alerts)
    return "\n".join(f"[{a['date']}] ({a['severity']}) {a['title']}: {a['detail']}" for a in alerts) or "No alerts."

@function_tool
def summarize_service_alerts(city: str, max_alerts: int = 3) -> str:
    """Summarise the recent service alerts for a city into key customer-facing insights."""
    alerts = _search_alerts(city, max_alerts)
    prompt = f"Summarise these service alerts for a customer-care team lead, with impact on customers: {alerts}"
    return client.responses.create(model="gpt-4.1", input=prompt).output_text

alerts_agent = Agent(
    name="Service Alerts Agent",
    instructions="Report current service disruptions using only the tools provided.",
    model="gpt-4o-mini",
    tools=[search_service_alerts],
)
result = await Runner.run(alerts_agent, "Are there any service disruptions affecting customers in Delhi?")
print(result.final_output)

In [ ]:
result = await Runner.run(alerts_agent, "Give me the latest alerts for Pune and Chennai.")
print(result.final_output)

In [ ]:
alerts_summary_agent = Agent(
    name="Service Alerts Summary Agent",
    instructions="Give a summary of current service disruptions using only the tools provided.",
    model="gpt-4o-mini",
    tools=[summarize_service_alerts],
)
result = await Runner.run(alerts_summary_agent, "Summarise the disruptions affecting Delhi customers.")
print(result.final_output)

## What changed?

| | Plain LLM | Agent with tools |
|---|---|---|
| Knows today's order status | No | Yes (calls the OMS) |
| Can invent an ETA | Yes | Much less likely: instructed to use tool data only |
| Auditable | No | Yes: every tool call and result is visible |

## Discussion (MBA class)
1. "Where is my order?" (WISMO) is often 20-30% of contact volume. What is the business case for automating it?
2. Which tools would *you* give a first-line care agent, and which actions (refunds, cancellations) would you **not** give it? Why?
3. The model chose the argument `ORD-78120` from free text. What could go wrong, and how would you validate it?